# CLIF 2.1 → 3.0 Column Crosswalk — Demo

This workbook demonstrates `clifpy.crosswalk_table_2_1_to_3_0`, which migrates a site's
**standardized** columns (`*_category`, `*_group`, `*_type`) from CLIF **2.1** to **3.0**.

What it does:
- Lowercases + snake_cases values (`IMV` → `imv`, `Black or African American` → `black_or_african_american`, `l&d` → `l_and_d`).
- Applies curated, non-derivable renames (`High Flow NC` → `hfnc`, `Assist Control-Volume Control` → `acvc`, `Long Term Care Hospital (LTACH)` → `ltach`, `Psychiatric Hospital` → `mental_health_hosp`).
- Leaves un-mappable values **unchanged** and flags them in a report (1→many like `albumin`, or values removed in 3.0 like the `csf` lab specimen).

It is **non-mutating**: it returns a converted *copy* plus a change report. The default table scope is the 16 CLIF beta tables.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import pandas as pd

from clifpy import crosswalk_table_2_1_to_3_0, normalize_category_value, BETA_TABLES
from clifpy.data import load_demo_clif

print("Beta tables (default scope):")
print(", ".join(BETA_TABLES))

Beta tables (default scope):
patient, hospitalization, adt, vitals, labs, patient_assessments, medication_admin_continuous, medication_admin_intermittent, respiratory_support, position, patient_procedures, code_status, crrt_therapy, hospital_diagnosis, microbiology_culture, microbiology_susceptibility


In [2]:
# The built-in demo data is in CLIF 2.1 format.
co = load_demo_clif(tables=['patient', 'hospitalization', 'adt', 'respiratory_support', 'vitals'])
print("Loaded demo tables:", co.get_loaded_tables())

📢 ClifOrchestrator initialized


📢 Initialized patient table


📢 Data directory: /Users/dema/WD/clifpy/clifpy/data/clif_demo


📢 File type: parquet


📢 Timezone: UTC


...

📢 Timezone: UTC


📢 Output directory: /Users/dema/WD/clifpy/examples/output


📢 Loaded schema for 'vitals' (CLIF 2.1)


📢 Loaded outlier configuration


Loaded demo tables: ['patient', 'hospitalization', 'adt', 'vitals', 'respiratory_support']


## 1. `respiratory_support` — abbreviations + snake_case

`device_category` and `mode_category` carry the hardest renames (`High Flow NC` → `hfnc`,
`Assist Control-Volume Control` → `acvc`, `Pressure Support/CPAP` → `ps_or_cpap`).

In [3]:
rs = co.respiratory_support.df
rs_30, rs_report = crosswalk_table_2_1_to_3_0(rs, "respiratory_support")

for col in ["device_category", "mode_category"]:
    before = sorted(rs[col].dropna().unique())
    mapping = {v: normalize_category_value(v) for v in before}  # display only
    pairs = pd.DataFrame({
        "2.1 value": before,
        "3.0 value": [rs_30.loc[rs[col] == v, col].iloc[0] if (rs[col] == v).any() else None for v in before],
    })
    print(f"\n=== {col} ===")
    print(pairs.to_string(index=False))

print("\nReport is_complete:", rs_report["is_complete"])
print("device_category values converted:", rs_report["columns"]["device_category"]["n_values_converted"])


=== device_category ===
    2.1 value     3.0 value
         CPAP          cpap
    Face Mask     face_mask
 High Flow NC          hfnc
          IMV           imv
        NIPPV         nippv
Nasal Cannula nasal_cannula
        Other         other

=== mode_category ===
                        2.1 value        3.0 value
    Assist Control-Volume Control             acvc
                            Other            other
                 Pressure Control pressure_control
            Pressure Support/CPAP       ps_or_cpap
Pressure-Regulated Volume Control             prvc
                             SIMV             simv
                   Volume Support   volume_support

Report is_complete: True
device_category values converted: 2542


## 2. `hospitalization.discharge_category` — curated renames

`Long Term Care Hospital (LTACH)` → `ltach`, `Skilled Nursing Facility (SNF)` → `snf`,
`Psychiatric Hospital` → `mental_health_hosp`, `Against Medical Advice (AMA)` → `ama`.

In [4]:
hosp = co.hospitalization.df
hosp_30, hosp_report = crosswalk_table_2_1_to_3_0(hosp, "hospitalization")

before = sorted(hosp["discharge_category"].dropna().unique())
out = pd.DataFrame({
    "2.1 value": before,
    "3.0 value": [hosp_30.loc[hosp["discharge_category"] == v, "discharge_category"].iloc[0] for v in before],
})
print(out.to_string(index=False))
print("\nis_complete:", hosp_report["is_complete"])

                      2.1 value          3.0 value
            Acute Care Hospital    acute_care_hosp
 Acute Inpatient Rehab Facility     acute_ip_rehab
   Against Medical Advice (AMA)                ama
                        Expired            expired
                           Home               home
                        Hospice            hospice
Long Term Care Hospital (LTACH)              ltach
                        Missing            missing
           Psychiatric Hospital mental_health_hosp
 Skilled Nursing Facility (SNF)                snf

is_complete: True


## 3. `patient` — demographics snake_cased

In [5]:
pat = co.patient.df
pat_30, pat_report = crosswalk_table_2_1_to_3_0(pat, "patient")
for col in ["race_category", "sex_category", "ethnicity_category"]:
    if col in pat.columns:
        b = sorted(pat[col].dropna().unique())
        a = [pat_30.loc[pat[col] == v, col].iloc[0] for v in b]
        print(f"{col}: " + ", ".join(f"{x!r}->{y!r}" for x, y in zip(b, a)))

race_category: 'Black or African American'->'black_or_african_american', 'Other'->'other', 'Unknown'->'unknown', 'White'->'white'
sex_category: 'Female'->'female', 'Male'->'male'
ethnicity_category: 'Hispanic'->'hispanic', 'Non-Hispanic'->'non_hispanic', 'Unknown'->'unknown'


## 4. The change report

Every call returns a structured report: per-column `n_values_converted`, `ambiguous`
(left-unchanged 1→many), `unresolved` (produced a value not in the 3.0 list), and an
overall `is_complete`.

In [6]:
print(json.dumps(rs_report, indent=2, default=str))

{
  "table": "respiratory_support",
  "from_version": "2.1",
  "to_version": "3.0",
  "columns": {
    "device_category": {
      "n_values_converted": 2542,
      "ambiguous": [],
      "unresolved": []
    },
    "mode_category": {
      "n_values_converted": 1452,
      "ambiguous": [],
      "unresolved": []
    }
  },
  "is_complete": true
}


## 5. Edge cases — ambiguous & removed values (crafted sample)

The demo data doesn't contain these, so we craft a tiny frame to show how the crosswalk
flags values it will **not** silently convert:
- `albumin` → 1→many (`albumin_5` / `albumin_25`): left unchanged, reported as **ambiguous**.
- `csf` lab specimen: **removed** in 3.0, no equivalent: left unchanged, reported.

In [7]:
# Ambiguous 1->many: albumin in medication_admin_continuous
med = pd.DataFrame({"hospitalization_id": ["H1", "H2"],
                    "med_category": ["albumin", "norepinephrine"]})
med_30, med_report = crosswalk_table_2_1_to_3_0(med, "medication_admin_continuous")
print("med_category after:", list(med_30["med_category"]), "(albumin left unchanged)")
print("ambiguous:", json.dumps(med_report["columns"]["med_category"]["ambiguous"], default=str))

# Removed value: csf lab specimen
labs = pd.DataFrame({"hospitalization_id": ["H1", "H2", "H3"],
                     "lab_specimen_category": ["blood/plasma/serum", "csf", "urine"]})
labs_30, labs_report = crosswalk_table_2_1_to_3_0(labs, "labs")
print("\nlab_specimen_category:", list(labs["lab_specimen_category"]), "->", list(labs_30["lab_specimen_category"]))
print("is_complete:", labs_report["is_complete"])
print("ambiguous:", json.dumps(labs_report["columns"]["lab_specimen_category"]["ambiguous"], default=str))

med_category after:

 ['albumin', 'norepinephrine'] (albumin left unchanged)
ambiguous: [{"original": "albumin", "candidates": ["albumin_5", "albumin_25"], "reason": "Concentration-dependent split in 3.0; choose albumin_5 or albumin_25 from the product concentration.", "count": 1}]



lab_specimen_category:

 ['blood/plasma/serum', 'csf', 'urine'] -> ['plasma_blood', 'csf', 'urine']
is_complete: False
ambiguous: [{"original": "csf", "candidates": [], "reason": "Removed from lab_specimen_category in CLIF 3.0; no direct equivalent.", "count": 1}]


## 6. Non-mutating, and the normalizer

The input frame is never modified, and `normalize_category_value` is the deterministic
transform used for everything that isn't a curated rename.

In [8]:
rs_before = co.respiratory_support.df.copy()
_ = crosswalk_table_2_1_to_3_0(co.respiratory_support.df, "respiratory_support")
print("input unchanged after crosswalk:", co.respiratory_support.df.equals(rs_before))

for v in ["IMV", "Non-Hispanic", "l&d", "DNR/DNI", "Mobility/Activity", "pulmonary vasodilators (IV)"]:
    print(f"  {v!r:38} -> {normalize_category_value(v)!r}")

input unchanged after crosswalk: True
  'IMV'                                  -> 'imv'
  'Non-Hispanic'                         -> 'non_hispanic'
  'l&d'                                  -> 'l_and_d'
  'DNR/DNI'                              -> 'dnr_or_dni'
  'Mobility/Activity'                    -> 'mobility_or_activity'
  'pulmonary vasodilators (IV)'          -> 'pulmonary_vasodilators_iv'
